
# Python 核心特性：yield、生成器表达式与惰性求值

作为 Java 开发者，你可以把生成器理解为**“可以暂停和恢复的函数”**。它完美诠释了 Python 的**惰性求值（Lazy Evaluation）**哲学。

## 一、什么是惰性求值机制？（核心思想）

**惰性求值**，通俗地说就是**“按需生产，绝不提前浪费”**。

- **传统思维（急切求值）**：就像工厂一次性生产 1000 个零件，全部堆在仓库（内存）里，等你慢慢用。如果数据量极大，仓库直接爆掉（OOM 内存溢出）。
- **惰性求值**：工厂不建仓库，而是建了一条流水线。你拿走一个，它就现场生产一个；你不拿，它就不生产。内存占用永远只有一件物品的大小。

在 Python 中，`map()`、`filter()` 以及 `yield` 生成器，底层都是基于惰性求值机制实现的。

## 二、yield 关键字：生成器的灵魂

在 Python 中，**只要函数体内包含 `yield` 关键字，这个函数就不再是普通函数，而是一个生成器函数（Generator Function）**。调用它时，不会执行函数体，而是返回一个生成器对象（迭代器）。

> **Java 对比**：Java 中没有直接等价物，但逻辑上类似于你手动实现了一个 `Iterator` 接口，在 `next()` 方法里写状态机。

### 代码示例

```python
def count_up_to(max):
    n = 1
    while n <= max:
        print(f"正在生产数字: {n}")
        yield n  # 暂停函数，并把 n 返回给调用者
        n += 1

# 1. 调用生成器函数，此时代码【根本没有执行】
gen = count_up_to(3) 
print(type(gen))  # <class 'generator'>

# 2. 每次调用 next()，函数从上次 yield 的地方【恢复执行】
print(next(gen))  # 输出: 正在生产数字: 1 \n 1
print(next(gen))  # 输出: 正在生产数字: 2 \n 2
print(next(gen))  # 输出: 正在生产数字: 3 \n 3
print(next(gen))  # 抛出 StopIteration 异常，表示数据耗尽
```

### `yield` vs `return` 的本质区别

- **`return`**：函数彻底结束，销毁所有局部变量，返回值。
- **`yield`**：函数**暂停（挂起）**，保留所有的局部变量和执行状态，返回值。下次调用时，从 `yield` 的下一行继续执行。

## 三、生成器表达式：极简的惰性列表

还记得我们学过的**列表推导式**吗？（例如 `[x*2 for x in range(10)]`）。
生成器表达式就是把列表推导式的**方括号 `[]` 换成圆括号 `()`**。

### 代码示例

```python
# 列表推导式：立刻生成包含 100万个元素的列表，极其耗费内存
list_comp = [x * 2 for x in range(1000000)] 

# 生成器表达式：只返回一个生成器对象，几乎不占内存
gen_exp = (x * 2 for x in range(1000000)) 

print(gen_exp)  # <generator object <genexpr> at 0x...>
```

**使用场景**：当你只需要遍历一次数据，而不需要像列表那样通过索引 `list[0]` 随机访问时，**永远优先使用生成器表达式**。

## 四、为什么生成器如此强大？（实战对比）

假设我们要读取一个 10GB 的日志文件，提取包含 "ERROR" 的行：

### ❌ 传统列表写法（内存爆炸）

```python
def get_errors(filename):
    errors = []
    with open(filename) as f:
        for line in f:
            if "ERROR" in line:
                errors.append(line)  # 10GB文件可能有几百万行错误，全部塞进内存
    return errors
```

### ✅ 生成器写法（内存恒定，极其优雅）

```python
def get_errors(filename):
    with open(filename) as f:
        for line in f:
            if "ERROR" in line:
                yield line  # 读到一行就吐出一行，内存永远只占一行的空间！
```

## 五、核心总结

1. **惰性求值**：Python 处理大数据的核心哲学，用时间换空间。
2. **yield**：让函数具备“记忆”能力，暂停并保存状态，是构建生成器的基石。
3. **生成器表达式**：用 `()` 替代 `[]`，是处理海量数据流的最简语法糖。
4. **黄金法则**：在 Python 中，**能用生成器，就绝不用列表**。

## 六、动手练习：无限斐波那契生成器

请尝试用 Python 实现以下需求，巩固今天学的知识：

1. 编写一个生成器函数 `fibonacci_gen()`。
2. 它不需要接收参数，能够**无限**生成斐波那契数列（0, 1, 1, 2, 3, 5, 8...）。
3. 使用 `while True` 和 `yield` 来实现。
4. 在外部调用时，结合 `for` 循环，打印出前 10 个斐波那契数（提示：可以用 `enumerate` 配合 `break` 来控制次数）。

In [1]:
# 请尝试用 Python 实现以下需求，巩固今天学的知识：

# 1. 编写一个生成器函数 `fibonacci_gen()`。
# 2. 它不需要接收参数，能够**无限**生成斐波那契数列（0, 1, 1, 2, 3, 5, 8...）。F(0)=0,F(1)=1 ，当 n>1 时， F(n)=F(n−1)+F(n−2) 。
# 3. 使用 `while True` 和 `yield` 来实现。
# 4. 在外部调用时，结合 `for` 循环，打印出前 10 个斐波那契数（提示：可以用 `enumerate` 配合 `break` 来控制次数）。

def fibonacci_gen():
    """
    无限生成斐波那契数列的生成器函数
    """
    a, b = 0, 1  # 初始化斐波那契数列的前两项
    while True:
        yield a  # 暂停函数，将当前的 a 返回给调用者
        a, b = b, a + b  # 状态更新：a 变成原来的 b，b 变成原来的 a+b

# --- 调用与测试 ---
# 结合 enumerate 和 break 提取前 10 个数字
for index, num in enumerate(fibonacci_gen()):
    if index >= 10:
        break
    print(f"第 {index + 1} 个斐波那契数: {num}")
        

第 1 个斐波那契数: 0
第 2 个斐波那契数: 1
第 3 个斐波那契数: 1
第 4 个斐波那契数: 2
第 5 个斐波那契数: 3
第 6 个斐波那契数: 5
第 7 个斐波那契数: 8
第 8 个斐波那契数: 13
第 9 个斐波那契数: 21
第 10 个斐波那契数: 34
